In [ ]:
# ==============================================================================
# 📦 CONFIGURATION INITIALE : Installation et Importation des Dépendances
# ==============================================================================
import os
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", pip, "install", "-q", package])

print("Installation des packages requis (transformers, torch, datasets)...")
install("transformers")
install("torch")
install("datasets")
install("pandas")

import torch
import torch.nn.functional as F
import pandas as pd
from transformers import (
    BertTokenizer, 
    pipeline, 
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AutoModelForTokenClassification
)
from IPython.display import display, HTML

print("Environnement initialisé avec succès.\n" + "="*50 + "\n")

# ==============================================================================
# 🌟 EXERCISE 1: Tokenization with BERT
# ==============================================================================
print("Exécutant l'Exercice 1 : Tokenization avec BERT...")

# 1. Chargement du tokenizer BERT
tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')
sentence = "Transformers and BERT models are game changers for Natural Language Processing."

# 2. Tokenization brute
tokens = tokenizer_bert.tokenize(sentence)
print("-> Tokens Bruts :", tokens)

# 3. Encodage complet avec tokens spéciaux, padding et troncature
inputs_bert = tokenizer_bert(
    sentence,
    add_special_tokens=True,
    max_length=16,
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

print("-> Token IDs (Numerical) :", inputs_bert['input_ids'][0].tolist())
print("-> Attention Mask        :", inputs_bert['attention_mask'][0].tolist())

# 4. Conversion inverse pour visualiser la structure finale
converted_tokens = tokenizer_bert.convert_ids_to_tokens(inputs_bert['input_ids'][0])
print("-> Tokens décodés avec structure :", converted_tokens)
print("\n" + "="*50 + "\n")


# ==============================================================================
# 🌟 EXERCISE 2: Sentiment Analysis with BERT Pipeline
# ==============================================================================
print("Exécutant l'Exercice 2 : Sentiment Analysis via Pipeline...")

# 1. Initialisation de la pipeline d'analyse de sentiment
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sample_text = "I absolutely love exploring new architectures in deep learning, it feels like magic!"
result_pipe = sentiment_pipeline(sample_text)[0]

print(f"-> Texte : '{sample_text}'")
print(f"-> Résultat : {result_pipe['label']} (Score de confiance : {result_pipe['score']:.4f})")
print("\n" + "="*50 + "\n")


# ==============================================================================
# 🌟 EXERCISE 3: Building a Custom Sentiment Analyzer
# ==============================================================================
print("Exécutant l'Exercice 3 : Custom Sentiment Analyzer Class...")

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()

    def preprocess(self, text):
        return self.tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    def predict(self, text):
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1).squeeze().tolist()
        label_id = torch.argmax(logits, dim=-1).item()
        
        # Mapping spécifique au jeu de données SST-2 (0: NEGATIVE, 1: POSITIVE)
        label = "POSITIVE" if label_id == 1 else "NEGATIVE"
        confidence = probabilities[label_id]
        
        return {"label": label, "confidence": confidence}

# Test du composant personnalisé
analyzer = BERTSentimentAnalyzer()
test_phrases = [
    "The assignment was quite difficult and confusing at first.",
    "The documentation provided made the setup incredibly straightforward!"
]

for phrase in test_phrases:
    pred = analyzer.predict(phrase)
    print(f"-> Texte : '{phrase}' -> {pred['label']} ({pred['confidence']:.4f})")
print("\n" + "="*50 + "\n")


# ==============================================================================
# 🌟 EXERCISE 4: Understanding BERT for Named Entity Recognition (NER)
# ==============================================================================
print("Exécutant l'Exercice 4 : Named Entity Recognizer (NER) Class...")

class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dbmdz/bert-large-cased-finetuned-conll03-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.eval()
        self.labels_map = self.model.config.id2label

    def recognize_entities(self, text):
        inputs = self.tokenizer(text, return_tensors="pt")
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            
        predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()
        
        extracted_entities = []
        for token, pred_id in zip(tokens, predictions):
            label = self.labels_map[pred_id]
            if label != 'O' and token not in ['[CLS]', '[SEP]']:
                extracted_entities.append((token, label))
        return extracted_entities

# Test du composant NER
ner_recognizer = BERTNamedEntityRecognizer()
ner_text = "Alice went to Paris last July to attend a conference hosted by Google."
entities = ner_recognizer.recognize_entities(ner_text)

print(f"-> Texte : '{ner_text}'")
print("-> Entités détectées :", entities)
print("\n" + "="*50 + "\n")


# ==============================================================================
# 🌟 EXERCISE 5: Comparing BERT and GPT (Affichage HTML)
# ==============================================================================
print("Exécutant l'Exercice 5 : Affichage du Tableau Comparatif BERT vs GPT...")

data_comp = {
    "Critère": ["Architecture", "Sens de lecture", "Objectif Principal", "Cas d'usage types", "Forces", "Faiblesses"],
    "BERT": [
        "Encoder-only (Transformer)", 
        "Bidirectionnel (analyse globale simultanée)", 
        "Compréhension du langage (NLU)", 
        "Classification, Analyse de sentiment, NER, Extraction de réponses", 
        "Compréhension contextuelle holistique ultra-précise", 
        "Incapable de générer du texte de manière itérative ou fluide"
    ],
    "GPT": [
        "Decoder-only (Transformer)", 
        "Unidirectionnel (de gauche à droite auto-regressif)", 
        "Génération de contenu (NLG)", 
        "Restauration de texte, Chatbots, Génération de code, Création d'histoires", 
        "Excellente fluidité narrative sur de longues séquences", 
        "Vision asymétrique/restreinte pour l'extraction brute de faits"
    ]
}

df_comp = pd.DataFrame(data_comp)
display(HTML(df_comp.to_html(index=False)))

print("\nRéflexion théorique :")
print("BERT (Analyseur) utilise le Masked Language Modeling pour capter l'environnement complet d'un mot.")
print("GPT (Conteur) se base sur le Causal Language Modeling pour prédire logiquement le token suivant.")
print("\n" + "="*50 + "\n")


# ==============================================================================
# 🌟 EXERCISE 6: Exploring BERT Applications in Retrieval-Augmented Generation (RAG)
# ==============================================================================
print("Exécutant l'Exercice 6 : Synthèse sur les architectures RAG...")

rag_explanation = """
Le framework RAG (Retrieval-Augmented Generation) s'appuie sur deux piliers :

1. Génération d'Embeddings (Le Rôle de BERT) :
   BERT convertit les bases de connaissances (PDFs, Wikis) et les requêtes des utilisateurs 
   en vecteurs denses. Grâce à sa nature bidirectionnelle, ces embeddings capturent le sens 
   sémantique profond plutôt que de simples mots-clés.

2. Base de Données Vectorielle :
   Les vecteurs générés par BERT sont stockés dans un index vectoriel (ex: Chroma, Pinecone). 
   Lorsqu'une question est posée, le système effectue une recherche de similarité cosinus 
   pour extraire uniquement les segments de documents pertinents.

3. Synergie BERT + GPT (Exemple concret) :
   - Requête : 'Quels sont les effets secondaires du médicament X ?'
   - Phase 1 (Retrieval - BERT) : Génère le vecteur de la question, interroge la base et 
     extrait 2 paragraphes ultra-ciblés issus d'une notice médicale complète.
   - Phase 2 (Generation - GPT) : Reçoit le prompt enrichi : 'En utilisant ces faits [Fragments BERT], 
     réponds à la question'. GPT formule alors une réponse fluide, naturelle et véridique.
"""
print(rag_explanation)
print("="*50 + "\nFin de l'exécution complète des exercices.")